In [ ]:
!pip install distopf==1.1.0.dev0

In [ ]:
!wget -N -q "https://matematica.unipv.it/gualandi/solvers/ipopt-linux64.zip"

In [ ]:
!unzip -o -q ipopt-linux64

In [ ]:
import os; os.environ['PATH'] += ':/content/ipopt'

In [37]:
from distopf import create_case, CASES_DIR
import distopf as opf

# Objective Functions
The following objectives are used in this notebook.

## Loss Minimization Objective

The following objective minimizes total system losses:

$$\min \sum_{t \in \mathcal{T}} \sum_{i,j,p} \left[\left(P^{pp}_{ij}(t)\right)^2 + \left(Q^{pp}_{ij}(t)\right)^2\right] r^{pp}_{ij}$$

Where:

- $P^{pp}_{ij}(t)$ is the active power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.p_flow[i, j, p, t]`)
- $Q^{pp}_{ij}(t)$ is the reactive power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.q_flow[i, j, p, t]`)
- $r^{pp}_{ij}$ is the resistance of branch from bus $i$ to bus $j$ on phase $p$ (`model.r[i, j, p+p]`)
- $i,j,p$ represents all branches and phases
- $\mathcal{T}$ is the set of time steps (`model.time_set`)

## Cost Minimization Objective

The following objective minimizes the total energy procurement cost from the swing bus over the 24-hour period:

$$\min \sum_{t \in \mathcal{T}} \sum_{i,j,p \in \text{swing}} P^{pp}_{ij}(t) \cdot \text{price}(t) \cdot \Delta t$$

Where:

- $P^{pp}_{ij}(t)$ is the active power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.p_flow[i, j, p, t]`)
- $\text{price}(t)$ is the electricity price at time $t$ ($/p.u. power) (`model.price[t]`)
- $\Delta t$ is the duration of each time step in hours (`model.delta_t`)
- The sum is over all branches originating from swing buses (substation buses)
- $\mathcal{T}$ is the set of time steps (`model.time_set`)

## Voltage Deviation Minimization Objective

The following objective minimizes voltage deviation from nominal (1.0 p.u.):

$$\min \sum_{t \in \mathcal{T}} \sum_{i,p} \left(v^2_{i,p}(t) - 1.0\right)^2$$

Where:

- $v^2_{i,p}(t)$ is the squared voltage magnitude at bus $i$ on phase $p$ at time $t$ (`model.v2[i, p, t]`)
- The objective penalizes deviations of squared voltage from nominal squared voltage (1.0)
- This is useful for voltage regulation and supporting grid stability
- $i,p$ represents all buses and phases (`model.bus_phase_set`)
- $\mathcal{T}$ is the set of time steps (`model.time_set`)

# Multi-Period Optimal Power Flow (OPF)

## Create and run a 24 hour multi-period optimal power flow on the IEEE 123 bus system with DERs on 5 buses and 2 batteries.
Using the LinDistFlow formulation. This problem has linear constraints and a linear objective function.

The objective here is to minimize the total cost over the 24 hour period. 

In [ ]:
case = create_case(CASES_DIR / "csv" / "ieee123_bat", start_step=0, n_steps=24)

First lets look at the schedules used for this case. The schedules file contains any time varying data for the case, such as load profiles, PV generation profiles. In this case, there are three columns, "PV" for the generation profiles, "default" for the load profiles, and "price" for the electricity price. The first two columns are normalized to 1, and the last column is in $ per p.u. power (MW in this case).

In [4]:
case.plot_schedules()

Now let's run the optimal power flow and look at the results.

In [5]:

result = case.run_opf(objective="cost", control_regulators=True, control_capacitors=True)

result.plot_voltages().show()
result.plot_network().show()
result.plot_batteries().show()

# Modeling Split Phase Triplex Lines.

In [6]:
case = create_case(CASES_DIR / "csv"/ "triplex_pv")
result = case.run_opf(objective="loss")
result.plot_network().show()
result.plot_voltages().show()
result.plot_power_flows().show()

# Non-Linear "branchflow" Formulation
By default the linear "lindist" formulation is used for the optimal power flow. However, we can also use the non-linear "branchflow" formulation. This formulation is more accurate, but it is also more computationally expensive. Here we will show it on the 123-bus for single time step cases where we vary which control variable is used on the generators. The default is to control both P and Q, but we can also choose to control only P or only Q.

In [74]:
case = create_case(
    CASES_DIR / "csv" / "ieee123_30der", 
    ignore_schedule=True  # ignore the schedule and use the default values for the DERs and loads
    )
case.modify(gen_mult = 5)

We can tell the solver which variable to control on the generators. 
Options include "PQ", "P", "Q", and "".

In [96]:
case = create_case(
    CASES_DIR / "csv" / "ieee123_30der", 
    ignore_schedule=True  # ignore the schedule and use the default values for the DERs and loads
    )
case.modify(control_variable="PQ", gen_mult = 6)
result = case.run_opf(objective="voltage_deviation", formulation="branchflow", solver="ipopt")
print(result.objective_value)
opf.plot_polar(result.p_gens, result.q_gens)

0.1238151263397307


In [97]:
case = create_case(
    CASES_DIR / "csv" / "ieee123_30der", 
    ignore_schedule=True  # ignore the schedule and use the default values for the DERs and loads
    )
case.modify(control_variable="P", gen_mult = 6)
result = case.run_opf(objective="voltage_deviation", formulation="branchflow", solver="ipopt")
print(result.objective_value)
opf.plot_polar(result.p_gens, result.q_gens)

0.3471724211686981


In [100]:
case = create_case(
    CASES_DIR / "csv" / "ieee123_30der", 
    ignore_schedule=True  # ignore the schedule and use the default values for the DERs and loads
    )
case.modify(control_variable="Q", gen_mult = 6, v_max=1.06)  # In this case we need to either raise the voltage limit or lower the gen_mult to get a solution.
result = case.run_opf(objective="voltage_deviation", formulation="branchflow", solver="ipopt")
print(result.objective_value)
opf.plot_polar(result.p_gens, result.q_gens)

0.18953151536726268


# Mixed Integer Linear Programming (MILP)

Control of regulator taps or capacitor bank switches can be modeled as integer variables.

MINLP is also possible with an appropriate MINLP solver.

In [90]:
# First without Regulator or capacitor control as a baseline
case = create_case(CASES_DIR / "csv" / "ieee123_30der", ignore_schedule=True)
case.modify(control_variable="")  # turn off der control to highlight the effect of the regulators and capacitors on the voltage profile.
result = case.run_opf(objective=None, control_regulators=False, control_capacitors=False, formulation="lindist", solver="highs")
result.plot_voltages().show()

In [104]:
case = create_case(CASES_DIR / "csv" / "ieee123_30der", ignore_schedule=True)
case.modify(control_variable="")  # turn off der control to highlight the effect of the regulators and capacitors on the voltage profile.
result = case.run_opf(objective="voltage_deviation", control_regulators=True, control_capacitors=True, formulation="lindist", solver="ipopt")
result.plot_voltages().show()